# Demo 5.3 - HTTP FPGA Classification

本 Demo 演示通过 Central Server：

1. 上传 FPGA `.bit` 和 `.hwh`
2. 使用 `student_id` 和密码认证
3. 上传花卉图片
4. 调用 KV260 FPGA 进行分类
5. 获取分类结果

相关 Demo：

- `demo5_1`：FPGA/HLS 相关实验
- `demo5_2`：单块 KV260 上的 PYNQ + DMA FPGA 推理
- `demo5_3`：通过 Central Server HTTP 调用 FPGA

In [ ]:
from pathlib import Path
from getpass import getpass
import json
import time

import requests


In [ ]:
# 修改为本次实验使用的 Central Server、学生编号和文件路径。
CENTRAL = "http://192.168.31.254:8000"
STUDENT_ID = "student01"

BIT_FILE = Path("design_1_wrapper.bit")
HWH_FILE = Path("design_1_wrapper.hwh")
IMAGE_FILE = Path("flower.jpg")

POLL_INTERVAL_S = 1
WAIT_TIMEOUT_S = 120


In [ ]:
# 第一次用新的 student_id 上传 Artifact 时，此密码会成为该学生的密码。
# 以后使用同一个 student_id 时，必须继续输入相同密码。
PASSWORD = getpass("Student password: ")

if not 8 <= len(PASSWORD) <= 128:
    raise ValueError("Password must contain 8-128 characters.")


## 第一部分：上传 FPGA Artifact

In [ ]:
for label, path in (("BIT_FILE", BIT_FILE), ("HWH_FILE", HWH_FILE)):
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path.resolve()}")

if BIT_FILE.suffix.lower() != ".bit":
    raise ValueError(f"BIT_FILE must be a .bit file: {BIT_FILE}")
if HWH_FILE.suffix.lower() != ".hwh":
    raise ValueError(f"HWH_FILE must be a .hwh file: {HWH_FILE}")

print(f"BIT file: {BIT_FILE}")
print(f"HWH file: {HWH_FILE}")


In [ ]:
def parse_response(response, action):
    try:
        data = response.json()
    except ValueError as exc:
        raise RuntimeError(
            f"{action} returned HTTP {response.status_code} with a non-JSON response."
        ) from exc

    if response.ok:
        return data

    detail = data.get("detail", data) if isinstance(data, dict) else data
    if isinstance(detail, (dict, list)):
        detail = json.dumps(detail, ensure_ascii=False)

    if response.status_code == 401:
        message = "Authentication failed. Please check STUDENT_ID and password."
    elif response.status_code == 404 and action == "FPGA prediction":
        message = "No FPGA Artifact found for this student. Please upload .bit and .hwh first."
    elif response.status_code == 404:
        message = f"{action} resource was not found."
    elif response.status_code == 422:
        message = f"{action} was rejected. Check the student ID and selected files."
    elif response.status_code >= 500:
        message = "Central Server encountered an internal error."
    else:
        message = f"{action} failed."

    raise RuntimeError(f"{message}\nHTTP {response.status_code}: {detail}")


try:
    with BIT_FILE.open("rb") as bit_file, HWH_FILE.open("rb") as hwh_file:
        response = requests.post(
            f"{CENTRAL}/fpga/artifacts",
            data={
                "student_id": STUDENT_ID,
                "password": PASSWORD,
            },
            files={
                "bit": (BIT_FILE.name, bit_file),
                "hwh": (HWH_FILE.name, hwh_file),
            },
            timeout=120,
        )
except requests.Timeout as exc:
    raise RuntimeError("Artifact upload timed out after 120 seconds.") from exc
except requests.ConnectionError as exc:
    raise RuntimeError(f"Cannot connect to Central Server:\n{CENTRAL}") from exc
except requests.RequestException as exc:
    raise RuntimeError(f"Artifact upload request failed: {exc}") from exc

artifact_data = parse_response(response, "Artifact upload")

print("Artifact uploaded successfully\n")
print(f"HTTP status: {response.status_code}")
print(f"student_id : {artifact_data.get('student_id')}")
print(f"artifact_id: {artifact_data.get('artifact_id')}")
print(f"version    : {artifact_data.get('version')}")
print(f"status     : {artifact_data.get('status')}")


## 第二部分：上传图片进行 FPGA 分类

In [ ]:
if not IMAGE_FILE.is_file():
    raise FileNotFoundError(
        f"Test image not found: {IMAGE_FILE.resolve()}\n"
        "Place a .jpg, .jpeg, or .png image at IMAGE_FILE."
    )

IMAGE_CONTENT_TYPES = {
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".png": "image/png",
}
image_content_type = IMAGE_CONTENT_TYPES.get(IMAGE_FILE.suffix.lower())
if image_content_type is None:
    raise ValueError("IMAGE_FILE must be a .jpg, .jpeg, or .png file.")

print(f"Test image: {IMAGE_FILE}")


## 第三部分：提交请求并显示分类结果

如果 Central Server 返回 `202 queued`，下面的代码会每秒查询一次任务状态，最多等待 120 秒。

In [ ]:
def show_result(data):
    if data.get("status") != "completed":
        raise RuntimeError(
            f"FPGA request did not complete: {data.get('status')} - {data.get('error')}"
        )

    result = data.get("result")
    if not isinstance(result, dict):
        raise RuntimeError("Completed response does not contain a classification result.")

    flower_cn = result.get("flower_cn") or result.get("raw_class") or "-"
    flower_api = (
        result.get("flower_api")
        or result.get("flower")
        or result.get("predicted_class")
        or "-"
    )

    print("\n===== FPGA Classification Result =====\n")
    print(f"Flower     : {flower_cn}")
    print(f"API name   : {flower_api}")
    print(f"Class index: {result.get('class_index', '-')}")
    print(f"Confidence : {result.get('confidence', '-')}")


def wait_for_result(request_id):
    deadline = time.monotonic() + WAIT_TIMEOUT_S
    previous_status = None

    while time.monotonic() < deadline:
        remaining = deadline - time.monotonic()
        try:
            response = requests.get(
                f"{CENTRAL}/requests/{request_id}",
                headers=AUTH_HEADERS,
                timeout=min(10, max(1, remaining)),
            )
        except requests.Timeout as exc:
            raise RuntimeError("Request status query timed out.") from exc
        except requests.ConnectionError as exc:
            raise RuntimeError(f"Cannot connect to Central Server:\n{CENTRAL}") from exc
        except requests.RequestException as exc:
            raise RuntimeError(f"Request status query failed: {exc}") from exc

        data = parse_response(response, "Request status query")
        status = data.get("status")
        if status != previous_status:
            print(f"Request status: {status}")
            previous_status = status

        if status == "completed":
            return data
        if status == "failed":
            raise RuntimeError(f"FPGA request failed: {data.get('error')}")
        if status not in {"queued", "running"}:
            raise RuntimeError(f"Unexpected request status: {status}")

        time.sleep(min(POLL_INTERVAL_S, max(0, deadline - time.monotonic())))

    raise TimeoutError(f"FPGA request did not finish within {WAIT_TIMEOUT_S} seconds.")


In [ ]:
AUTH_HEADERS = {"X-Student-Password": PASSWORD}

try:
    with IMAGE_FILE.open("rb") as image_file:
        response = requests.post(
            f"{CENTRAL}/predict",
            headers=AUTH_HEADERS,
            data={"student_id": STUDENT_ID},
            files={
                "image": (IMAGE_FILE.name, image_file, image_content_type),
            },
            timeout=120,
        )
except requests.Timeout as exc:
    raise RuntimeError("FPGA prediction timed out after 120 seconds.") from exc
except requests.ConnectionError as exc:
    raise RuntimeError(f"Cannot connect to Central Server:\n{CENTRAL}") from exc
except requests.RequestException as exc:
    raise RuntimeError(f"FPGA prediction request failed: {exc}") from exc

prediction_data = parse_response(response, "FPGA prediction")

if response.status_code == 202 or prediction_data.get("status") in {"queued", "running"}:
    request_id = prediction_data.get("request_id")
    if not request_id:
        raise RuntimeError("Queued response does not contain request_id.")
    prediction_data = wait_for_result(request_id)

show_result(prediction_data)
